# পাঠ ০৪ - টুল ব্যবহার নকশা প্যাটার্ন

এই পাঠে আপনি মাইক্রোসফ্ট এজেন্ট ফ্রেমওয়ার্ক (পাইথন) ব্যবহার করে AI এজেন্টদের জন্য **টুল ব্যবহার** ডিজাইন প্যাটার্ন শিখবেন। আমরা আলোচনা করব:

- `@tool` ডেকোরেটর এবং টাইপ করা প্যারামিটার সহ ফাংশন টুল নির্ধারণ
- মডেলকে প্রতিটি টুলের কার্যক্রম জানাতে টুল স্কিমা প্রদান
- `approval_mode` দিয়ে টুল কার্যক্রম নিয়ন্ত্রণ
- পিড্যান্টিক মডেল এবং `response_format` এর মাধ্যমে **কাঠামোবদ্ধ আউটপুট** প্রদান

দৃশ্যপট হলো একটি **ভ্রমণ বুকিং এজেন্ট** যা গন্তব্যগুলি অনুসন্ধান করতে, প্রাপ্যতা পরীক্ষা করতে এবং ফ্লাইট তথ্য সংগ্রহ করতে পারে।


## সেটআপ


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## @tool ডেকোরেটর দিয়ে টুল সংজ্ঞায়িত করা

`@tool` ডেকোরেটর একটি সাধারণ Python ফাংশনকে এমন একটি টুলে রূপান্তরিত করে যা একটি এজেন্ট কল করতে পারে।
মূল পয়েন্টগুলি:

- **ডকস্ট্রিং** টুলের বিবরণ হয়ে যায় যা মডেল দেখে।
- **টাইপ অ্যানোটেশন** (বর্ণনা সহ `Annotated` সহ) টুল স্কিমা নির্ধারণ করে।
- `approval_mode` নিয়ন্ত্রণ করে ব্যবহারকারীকে প্রতিটি কলের আগে অনুমোদন দিতে হবে কিনা।


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## একাধিক টুল দিয়ে একটি এজেন্ট তৈরি করা

মডেলটি ব্যবহারকারীর প্রশ্নের উত্তর দিতে যেকোনো টুল ব্যবহার করতে পারে, সে জন্য ক্লায়েন্টকে তিনটি টুলই পাঠান।


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = client.as_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

## সরঞ্জামসহ গঠনবদ্ধ আউটপুট

`response_format` কে একটি Pydantic মডেলে সেট করার মাধ্যমে, এজেন্টকে বাধ্য করা হয় একটি সঠিক টাইপযুক্ত JSON অবজেক্ট ফেরত দিতে, মুক্ত-ফর্মের টেক্সটের পরিবর্তে। এটি তখন উপকারী যখন পরবর্তী কোড ফলাফলটি প্রোগ্রাম্যাটিকভাবে ব্যবহার করতে হয়।


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = client.as_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

## টুল অনুমোদন প্যাটার্নগুলি

`@tool` এর উপর `approval_mode` প্যারামিটার নিয়ন্ত্রণ করে টুল কলগুলি বাস্তবায়নের আগে মানব অনুমোদনের প্রয়োজন কিনা:

| মোড | আচরণ |
|---|---|
| `"never_require"` | টুল স্বয়ংক্রিয়ভাবে চলে — ব্যবহারকারীর নিশ্চিতকরণ প্রয়োজন নেই। |
| `"always_require"` | প্রতিটি কল কার্যকর হওয়ার আগে ব্যবহারকারীর দ্বারা অনুমোদিত হতে হবে। |

এমন টুলগুলির জন্য `"always_require"` ব্যবহার করুন যাদের পার্শ্বপ্রতিক্রিয়া রয়েছে (যেমন, একটি ফ্লাইট বুকিং করা, ক্রেডিট কার্ড চার্জ করা) যাতে একজন মানুষ লুপে থাকে।


In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

## সারমর্ম

এই পাঠে আপনি শিখেছেন কীভাবে:

১. **@tool** ডেকোরেটর ব্যবহার করে টাইপ করা প্যারামিটার এবং টুল স্কিমা হিসেবে কাজ করা ডকস্ট্রিং সহ টুল সংজ্ঞায়িত করবেন।
২. **একাধিক টুল সংযোগ করবেন** যাতে এজেন্ট ধারাবাহিকভাবে এগুলো কল করে জটিল প্রশ্নের উত্তর দিতে পারে।
৩. **গঠিত আউটপুট ফিরিয়ে দেবেন** `response_format` হিসেবে Pydantic মডেল ব্যবহার করে।
৪. সংবেদনশীল কার্যক্রমের জন্য মানুষের অনুমোদন রাখতে `approval_mode` সহ টুল অনুমোদন নিয়ন্ত্রণ করবেন।

এই প্যাটার্নগুলো নির্ভরযোগ্য, প্রোডাকশন-রেডি এজেন্ট তৈরি করার ভিত্তি গঠন করে যা বাহ্যিক সিস্টেমের সাথে নিরাপদে ইন্টারঅ্যাক্ট করতে পারে।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**অস্বীকৃতি**:
এই নথিটি AI অনুবাদ পরিষেবা [Co-op Translator](https://github.com/Azure/co-op-translator) ব্যবহার করে অনূদিত হয়েছে। যদিও আমরা শুদ্ধতার জন্য চেষ্টা করি, অনুগ্রহ করে মনে রাখবেন যে স্বয়ংক্রিয় অনুবাদে ত্রুটি বা অসঙ্গতি থাকতে পারে। মূল নথিটি তার স্বভাষায় কর্তৃত্বপূর্ণ উৎস হিসেবে বিবেচিত হওয়া উচিত। গুরুত্বপূর্ণ তথ্যের জন্য পেশাদার মানব অনুবাদ সুপারিশ করা হয়। এই অনুবাদের ব্যবহারে প্রয়োজনীয় ভুল বোঝাবুঝি বা ভুল ব্যাখ্যার জন্য আমরা দায়বদ্ধ নই।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
